## Find Company Table

In [3]:
import json
from pathlib import Path

PROCESSED = Path("../data/processed")


def load_doc(ticker):
	path = next(PROCESSED.glob(f"{ticker}_10-K_*.json"))
	return json.loads(path.read_text(encoding="utf-8"))


def find_tables(ticker, keyword, n=3, max_rows=15):
	"""Show the first n tables of a company that contain the keyword."""
	doc = load_doc(ticker)
	hits = [
		b for b in doc["blocks"]
		if b["type"] == "table" and keyword.lower() in b["content"].lower()
	]
	print(f"{ticker}: {len(hits)} tables containing {keyword!r}\n")
	for i, b in enumerate(hits[:n]):
		print(f"--- table {i} ---")
		print("CONTEXT:", b["context"][-200:])
		print("\n".join(b["content"].splitlines()[:max_rows]))
		print()


find_tables("JPM", "Net income", n=1)

JPM: 23 tables containing 'Net income'

--- table 0 ---
CONTEXT: Financial THREE-YEAR SUMMARY OF CONSOLIDATED FINANCIAL HIGHLIGHTS (unaudited) As of or for the year ended December 31, (in millions, except per share, ratio, employee data and where otherwise noted)
|  | 2025 Selected income statement data | 2024 Selected income statement data | 2023 Selected income statement data |
| --- | --- | --- | --- |
| Total net revenue | 182,447 | 177,556 | 158,104 |
| Total noninterest expense | 95,640 | 91,797 | 87,172 |
| Pre-provision profit (a) | 86,807 | 85,759 | 70,932 |
| Provision for credit losses | 14,212 | 10,678 | 9,320 |
| Income before income tax expense | 72,595 | 75,081 | 61,612 |
| Income tax expense | 15,547 | 16,610 | 12,060 |
| Net income | 57,048 | 58,471 | 49,552 |
| Earnings per share data |  |  |  |
| Net income:              Basic | 20.05 | 19.79 | 16.25 |
| Diluted | 20.02 | 19.75 | 16.23 |
| Average shares:     Basic | 2,776.5 | 2,873.9 | 2,938.6 |
| Diluted | 2,781.5 

## Evaluation Set Inspection

In [2]:
"""For each question, counts chunks from the same company that contain every evidence string.
Zero means the evidence is wrong."""
with open("../data/chunks.jsonl", encoding="utf-8") as f:
	chunks = [json.loads(line) for line in f]
with open("../eval/questions.jsonl", encoding="utf-8") as f:
	questions = [json.loads(line) for line in f if line.strip()]

problems = 0
for q in questions:
	if not q["evidence"]:
		continue
	matches = sum(
		c["ticker"] == q["ticker"]
		and all(e.lower() in c["text"].lower() for e in q["evidence"])
		for c in chunks
	)
	if matches == 0:
		problems += 1
		print("NO MATCH:", q["id"], q["evidence"])